In [1]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider, FloatSlider, fixed
DATA_DIR = "/home/tim-external/ros_ws/src/fsregistration/debug_results/2d/data"
print('Imports OK')

Imports OK


# 2D SOFFT Registration Analysis

Interactive notebook for analyzing 2D SOFFT registration results. Use the sliders to browse rotation candidates and correlation surfaces.

In [2]:
def load_csv(filename):
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        print(f'File not found: {filepath}')
        return None
    
    with open(filepath, 'r') as f:
        first_line = f.readline().strip()
    
    if not first_line:
        return None
    
    first_field = first_line.split('\t')[0] if '\t' in first_line else first_line.split()[0]
    is_header = not _is_numeric(first_field)
    
    if is_header:
        lines = []
        with open(filepath, 'r') as f2:
            next(f2)  # skip header
            for line in f2:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                # Skip the metadata row (6 columns) and section headers
                parts = line.split('\t')
                if len(parts) == 6 and _is_numeric(parts[0]):
                    continue  # skip metadata row like "128\t255\t..."
                if parts[0] in ('N', 'angleIndex'):
                    continue
                lines.append(line)
        if lines:
            delim = '\t' if '\t' in lines[0] else None
            return np.loadtxt(lines, delimiter=delim)
        return None
    
    return np.loadtxt(filepath)


def _is_numeric(s):
    try:
        float(s)
        return True
    except ValueError:
        return False


In [3]:
# --- Load all data ---
print('Loading data...')

# Input data
input_voxel1 = load_csv('voxelDataFFTW1.csv')
input_voxel2 = load_csv('voxelDataFFTW2.csv')
mag_fftw1 = load_csv('magnitudeFFTW1.csv')
phase_fftw1 = load_csv('phaseFFTW1.csv')
mag_fftw2 = load_csv('magnitudeFFTW2.csv')
phase_fftw2 = load_csv('phaseFFTW2.csv')
resampled1 = load_csv('resampledVoxel1.csv')
resampled2 = load_csv('resampledVoxel2.csv')

# Rotation data
rotation_corr = load_csv('rotationCorrelation1D.csv')
rotation_peaks = load_csv('rotationPeaks.csv')
if rotation_peaks is not None:
    rotation_peaks = np.atleast_2d(rotation_peaks)

# Metadata from dataForReadIn.csv
meta_filepath = os.path.join(DATA_DIR, 'dataForReadIn.csv')
data_meta = None
angle_data = None
if os.path.exists(meta_filepath):
    with open(meta_filepath, 'r') as f:
        lines = f.readlines()
    # Metadata row is line 1 (0-indexed): 128, 255, 0.1, 0.01, 17, 0
    if len(lines) > 1:
        meta_parts = lines[1].strip().split('\t')
        data_meta = {
            'N': int(meta_parts[0]),
            'correlationN': int(meta_parts[1]),
            'cellSize': float(meta_parts[2]),
            'potentialNecessaryForPeak': float(meta_parts[3]),
            'numAngles': int(meta_parts[4]),
            'numTotalSolutions': int(meta_parts[5]) if len(meta_parts) > 5 else 0
        }
    # Per-angle section starts after "angleIndex\tangle\tnumTranslations"
    angle_data = []
    for j, line in enumerate(lines):
        if 'angleIndex' in line:
            for k in range(j + 1, len(lines)):
                parts = lines[k].strip().split('\t')
                if len(parts) >= 3:
                    angle_data.append({
                        'angleIndex': int(parts[0]),
                        'angle': float(parts[1]),
                        'numTranslations': int(parts[2])
                    })
            break

# Infer N
if input_voxel1 is not None and input_voxel1.ndim == 1:
    N = int(round(input_voxel1.size ** 0.5))
    print(f'Inferred N = {N}')
else:
    N = 128
    print(f'Using default N = {N}')

correlationN = 2 * N - 1
print(f'correlationN = {correlationN}')

# Reshape 2D data
if input_voxel1 is not None:
    input_voxel1 = input_voxel1.reshape((N, N))
    input_voxel2 = input_voxel2.reshape((N, N))
    mag_fftw1 = mag_fftw1.reshape((N, N))
    phase_fftw1 = phase_fftw1.reshape((N, N))
    mag_fftw2 = mag_fftw2.reshape((N, N))
    phase_fftw2 = phase_fftw2.reshape((N, N))
    resampled1 = resampled1.reshape((N, N))
    resampled2 = resampled2.reshape((N, N))

print('All input data loaded.')
print(f'  Voxel shapes: ({N},{N})')
print(f'  Input1: min={input_voxel1.min():.4f}, max={input_voxel1.max():.4f}')
print(f'  Input2: min={input_voxel2.min():.4f}, max={input_voxel2.max():.4f}')
if data_meta:
    print(f'  Metadata: N={data_meta["N"]}, angles={data_meta["numAngles"]}, solutions={data_meta["numTotalSolutions"]}')
if angle_data:
    print(f'  Per-angle data: {len(angle_data)} angles')


Loading data...
Inferred N = 128
correlationN = 255
All input data loaded.
  Voxel shapes: (128,128)
  Input1: min=0.0000, max=0.4908
  Input2: min=0.0000, max=0.4956
  Metadata: N=128, angles=17, solutions=577
  Per-angle data: 17 angles


## Input Data Visualization

Side-by-side comparison of the two input frames and their FFT spectra.

## Resampled Fourier Magnitudes

The 2D Fourier magnitude of each image is projected onto the unit circle (S² in 2D).
These heatmaps show the resampled magnitude distributions for both input images.


In [4]:
# --- Resampled Fourier Magnitudes ---
fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Resampled Magnitude 1', 'Resampled Magnitude 2'))

fig.add_trace(
    go.Heatmap(z=resampled1, colorscale='Viridis', showscale=False),
    row=1, col=1
)
fig.add_trace(
    go.Heatmap(z=resampled2, colorscale='Viridis'),
    row=1, col=2
)

fig.update_layout(height=400, width=900, title_text='Resampled Fourier Magnitudes')
fig.update_xaxes(showticklabels=False, showgrid=False)
fig.update_yaxes(showticklabels=False, showgrid=False)
fig.show()


In [5]:
fig = make_subplots(rows=2, cols=2, 
    subplot_titles=('Voxel 1', 'Voxel 2', 'Magnitude FFT 1', 'Magnitude FFT 2'),
    specs=[[{'type': 'heatmap'}, {'type': 'heatmap'}],
           [{'type': 'heatmap'}, {'type': 'heatmap'}]])

fig.add_trace(go.Heatmap(z=input_voxel1, colorscale='Viridis', showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=input_voxel2, colorscale='Viridis'), row=1, col=2)
fig.add_trace(go.Heatmap(z=mag_fftw1, colorscale='Viridis', showscale=False), row=2, col=1)
fig.add_trace(go.Heatmap(z=mag_fftw2, colorscale='Viridis'), row=2, col=2)

fig.update_layout(height=700, width=700, title_text='Input Data & Spectra')
fig.update_xaxes(nticks=20)
fig.update_yaxes(nticks=20)
fig.show()

## Rotation Peaks

Detected rotation peaks (red x markers) overlaid on the correlation curve. Each peak shows the angle in radians and its index in the correlation array.

In [6]:
fig = go.Figure()

if rotation_corr is not None and rotation_corr.ndim == 2:
    indices = rotation_corr[:, 0]
    angles = rotation_corr[:, 1]
    correlations = rotation_corr[:, 2]
    fig.add_trace(go.Scatter(x=angles, y=correlations, mode='lines', name='Correlation', line=dict(width=2, color='steelblue')))
    fig.update_xaxes(title_text='Rotation Angle (rad)')
    fig.update_yaxes(title_text='Normalized Correlation')

if rotation_peaks is not None and rotation_peaks.ndim == 2:
    peak_angles = rotation_peaks[:, 0]
    peak_heights = rotation_peaks[:, 1]
    peak_indices = rotation_peaks[:, 3].astype(int)
    peak_text = [f'{a:.2f}rad (idx={idx})' for a, idx in zip(peak_angles, peak_indices)]
    fig.add_trace(go.Scatter(x=peak_angles, y=peak_heights, mode='markers+text',
                             marker=dict(size=12, color='red', symbol='x', line=dict(width=2)),
                             text=peak_text,
                             textposition='top center',
                             name='Detected Peaks'))

fig.update_layout(height=450, width=900, title_text='Rotation Peaks on Correlation Curve')
fig.show()

## Translation Correlation per Rotation Peak

For each detected rotation peak, the 1D translation correlation curve is computed.
Use the slider to select a rotation peak and inspect its translation correlation.
Red x markers indicate detected translation peaks for that rotation.


In [7]:
# --- Parse potentialTransformation files to build per-rotation translation data ---
import glob as glob_mod
import numpy.linalg as la

cellSize = data_meta['cellSize'] if data_meta else 1.0

def parse_transformation_matrix(filepath):
    '''Parse a potentialTransformation*.csv file and return rotation angle, translation, correlation.'''
    rows = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            try:
                rows.append([float(x) for x in parts])
            except ValueError:
                continue

    if len(rows) < 4:
        return None

    # First 4 rows = 4x4 homogeneous matrix
    matrix = np.array(rows[:4])
    R = matrix[:3, :3]
    t = matrix[:3, 3]

    # Extract rotation angle from 2D rotation matrix
    angle = np.arctan2(R[1, 0], R[0, 0])

    # Correlation score is on the line after the 4x4 matrix
    corr_score = None
    if len(rows) >= 5 and len(rows[4]) == 1:
        corr_score = rows[4][0]

    return {
        'angle': angle,
        'translation': t,
        'matrix': matrix,
        'correlation': corr_score,
        'translation_voxel_x': -t[0] / cellSize + (correlationN // 2) if cellSize != 0 else 0,
        'translation_voxel_y': -t[1] / cellSize + (correlationN // 2) if cellSize != 0 else 0,
    }


def find_closest_angle_index(angle, angle_list):
    '''Find the index of the rotation angle closest to the given angle.'''
    min_dist = float('inf')
    best_idx = 0
    for i, a in enumerate(angle_list):
        # Proper wrap-around normalization to [-pi, pi]
        diff = (angle - a + np.pi) % (2*np.pi) - np.pi
        dist = abs(diff)
        if dist < min_dist:
            min_dist = dist
            best_idx = i
    return best_idx
# Load all transformation files
trans_files = sorted(glob_mod.glob(os.path.join(DATA_DIR, 'potentialTransformation*.csv')))
print(f'Found {len(trans_files)} transformation files')

# Build mapping: rotation_angle_index -> list of translation solutions
if data_meta and angle_data:
    angle_list = [a['angle'] for a in angle_data]
    rotation_to_translations = {i: [] for i in range(len(angle_list))}
    rotation_to_corr_curve = {i: [] for i in range(len(angle_list))}

    for fpath in trans_files:
        basename = os.path.basename(fpath)
        # Extract number from filename
        num_str = basename.replace('potentialTransformation', '').replace('.csv', '')
        try:
            trans_num = int(num_str)
        except ValueError:
            continue

        trans_data = parse_transformation_matrix(fpath)
        if trans_data is None:
            continue

        angle = trans_data['angle']
        idx = find_closest_angle_index(angle, angle_list)
        rotation_to_translations[idx].append({
            'num': trans_num,
            'angle': angle,
            'translation': trans_data['translation'],
            'correlation': trans_data['correlation'],
            'translation_voxel_x': trans_data.get('translation_voxel_x'),
            'translation_voxel_y': trans_data.get('translation_voxel_y'),
        })

    # Sort each rotation's translations by correlation score (descending)
    for idx in rotation_to_translations:
        rotation_to_translations[idx].sort(key=lambda x: x['correlation'] if x['correlation'] is not None else 0, reverse=True)

    # Build correlation curves: for each rotation, collect correlation values
    for idx in rotation_to_translations:
        sol_list = rotation_to_translations[idx]
        rotation_to_corr_curve[idx] = [s['correlation'] if s['correlation'] is not None else 0 for s in sol_list]

    print(f'Rotation angles with solutions: {sum(1 for v in rotation_to_translations.values() if v)}')
    for idx in range(len(angle_list)):
        if rotation_to_translations[idx]:
            print(f'  Angle {angle_list[idx]:.4f} rad (idx={idx}): {len(rotation_to_translations[idx])} solutions')
else:
    rotation_to_translations = {}
    rotation_to_corr_curve = {}
    print('No angle data available for translation correlation')


# --- Interactive translation correlation plot ---
from ipywidgets import IntSlider, HBox, VBox, Label

num_rotation_peaks = len(angle_list) if data_meta and angle_data else 0

def plot_translation_correlation(peak_idx):
    if peak_idx >= len(angle_list) or not rotation_to_translations.get(peak_idx):
        fig = go.Figure()
        fig.add_annotation(text=f'No solutions for rotation index {peak_idx}',
                         xref='paper', yref='paper', showarrow=False,
                         font=dict(size=20))
        fig.update_layout(height=400, width=900)
        fig.show()
        return

    angle = angle_list[peak_idx]
    solutions = rotation_to_translations[peak_idx]
    corr_values = rotation_to_corr_curve[peak_idx]

    trans_indices = list(range(len(solutions)))

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=trans_indices,
        y=corr_values,
        mode='lines+markers',
        name='Correlation',
        line=dict(width=2, color='steelblue'),
        marker=dict(size=6),
    ))

    # Mark the best (highest correlation) as a peak
    if corr_values:
        best_idx = np.argmax(corr_values)
        best_val = corr_values[best_idx]
        fig.add_trace(go.Scatter(
            x=[best_idx],
            y=[best_val],
            mode='markers+text',
            marker=dict(size=14, color='red', symbol='x', line=dict(width=2)),
            text=['PEAK'],
            textposition='top center',
            name='Best Translation Peak',
        ))

    fig.update_layout(
        height=400, width=900,
        title_text=f'Translation Correlation — Rotation Peak {peak_idx} (angle={angle:.4f} rad, {len(solutions)} solutions)',
        xaxis_title_text='Translation Solution Index',
        yaxis_title_text='Correlation Score',
    )
    fig.show()


# Create interactive slider and plot via interact_manual
peak_slider = IntSlider(
    value=0,
    min=0,
    max=num_rotation_peaks - 1 if num_rotation_peaks > 0 else 0,
    step=1,
    description='Rotation Peak:',
    continuous_update=False,
    style={'description_width': 'initial'},
)

def on_peak_slider_interact(peak_idx):
    plot_translation_correlation(peak_idx)

from ipywidgets import interact_manual
interact_manual(on_peak_slider_interact, peak_idx=peak_slider)


Found 596 transformation files
Rotation angles with solutions: 17
  Angle 5.2033 rad (idx=0): 37 solutions
  Angle 5.5469 rad (idx=1): 29 solutions
  Angle 6.0377 rad (idx=2): 18 solutions
  Angle 0.0000 rad (idx=3): 24 solutions
  Angle 0.3927 rad (idx=4): 24 solutions
  Angle 0.5890 rad (idx=5): 35 solutions
  Angle 0.9817 rad (idx=6): 39 solutions
  Angle 1.4235 rad (idx=7): 33 solutions
  Angle 2.0126 rad (idx=8): 39 solutions
  Angle 2.1108 rad (idx=9): 29 solutions
  Angle 2.8471 rad (idx=10): 27 solutions
  Angle 3.1907 rad (idx=11): 40 solutions
  Angle 3.4361 rad (idx=12): 27 solutions
  Angle 3.6816 rad (idx=13): 22 solutions
  Angle 4.5160 rad (idx=14): 41 solutions
  Angle 4.7124 rad (idx=15): 86 solutions
  Angle 4.9578 rad (idx=16): 46 solutions


interactive(children=(IntSlider(value=0, continuous_update=False, description='Rotation Peak:', max=16, style=…

<function __main__.on_peak_slider_interact(peak_idx)>

## 2D Translation Correlation Surface

The 2D correlation 'hills' for the selected rotation angle.
Red 'x' markers indicate the detected translation peaks overlaid on the surface.


In [ ]:
# --- Data loading (runs once, outside @interact) ---
corr_file = os.path.join(DATA_DIR, f'translationCorrelation2D_angle0.csv')
corr_2d = np.loadtxt(corr_file, delimiter=',')
h, w = corr_2d.shape

# --- Get translation peaks ---
solutions = rotation_to_translations.get(0, [])
peak_x, peak_y, peak_z, peak_corr = [], [], [], []
for s in solutions:
    vx = s.get('translation_voxel_x')
    vy = s.get('translation_voxel_y')
    c = s.get('correlation')
    if vx is not None and vy is not None:
        ix = int(np.clip(np.round(vx), 0, w - 1))
        iy = int(np.clip(np.round(vy), 0, h - 1))
        peak_x.append(vx)
        peak_y.append(vy)
        peak_z.append(corr_2d[iy, ix])
        peak_corr.append(c if c is not None else 0)

max_peak_idx = max(0, num_rotation_peaks - 1) if data_meta and angle_data else 0

# --- Plotting function (called by @interact) ---
@interact(peak_idx=(0, max_peak_idx, 1))
def plot_2d_correlation_surface(peak_idx):
    fig = make_subplots(rows=1, cols=2,
        specs=[[{'type': 'heatmap'}, {'type': 'scene'}]],
        subplot_titles=(f'Correlation Heatmap — Peak {peak_idx}',
                         f'3D Surface — Peak {peak_idx}'))

    # Left: 2D heatmap with peak markers
    fig.add_trace(go.Heatmap(
        z=corr_2d, colorscale='Viridis', showscale=False
    ), row=1, col=1)
    if peak_x:
        fig.add_trace(go.Scatter(
            x=peak_x, y=peak_y, mode='markers',
            marker=dict(size=8, color='red', symbol='x', line=dict(width=2)),
            text=[f'{c:.3f}' for c in peak_corr],
            textposition='top center'
        ), row=1, col=1)

    # Right: 3D surface with peak markers
    fig.add_trace(go.Surface(
        z=corr_2d, colorscale='Viridis', showscale=True,
        colorbar_title='Correlation'
    ), row=1, col=2)
    if peak_x:
        fig.add_trace(go.Scatter3d(
            x=peak_x, y=peak_y, z=peak_z, mode='markers',
            marker=dict(size=4, color='red', symbol='cross',
                        line=dict(width=1, color='darkred'))
        ), row=1, col=2)

    fig.update_layout(
        height=600, width=1200,
        title_text=f'2D Translation Correlation — Peak {peak_idx} (angle=0.0000 rad, {len(solutions)} solutions)'
    )
    fig.update_xaxes(title_text='X Index', row=1, col=1)
    fig.update_yaxes(title_text='Y Index', row=1, col=1)
    fig.update_scenes(
        xaxis_title='X', yaxis_title='Y', zaxis_title='Correlation',
        aspectmode='cube', row=1, col=2
    )
    return fig


interactive(children=(IntSlider(value=8, description='peak_idx', max=16), Output()), _dom_classes=('widget-int…

## Blended Registration Results

For each rotation peak and each translation solution within that rotation,
the second voxel is transformed and blended with the first voxel.
Use the sliders to browse through rotation peaks and their translation solutions.
The blended image shows how well each candidate solution aligns the two scans.


In [9]:
# --- Load scipy for image transformation ---
try:
    from scipy.ndimage import affine_transform
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False
    print('scipy not available, using fallback with numpy')


def compute_2d_rotation_matrix(angle):
    '''Create a 2D rotation matrix from an angle in radians.'''
    c = np.cos(angle)
    s = np.sin(angle)
    return np.array([[c, -s], [s, c]])


def apply_transformation_to_voxel(voxel, matrix_4x4):
    '''Apply a 4x4 homogeneous transformation matrix to a 2D voxel.
    Returns the transformed voxel of the same shape.'''
    R = matrix_4x4[:2, :2]
    t = matrix_4x4[:2, 2]

    h, w = voxel.shape
    # Create grid of output coordinates
    yy, xx = np.mgrid[:h, :w]
    coords = np.vstack([xx.ravel(), yy.ravel(), np.ones(h * w)])

    # Transform coordinates
    transformed = R @ coords[:2] + t.reshape(2, 1)
    new_x = transformed[0].reshape(h, w)
    new_y = transformed[1].reshape(h, w)

    if HAS_SCIPY:
        # Use scipy for efficient interpolation
        # affine_transform expects the inverse mapping
        inv_R = np.linalg.inv(R)
        inv_t = -inv_R @ t
        affine_matrix = inv_R
        offset = -inv_t
        result = affine_transform(
            voxel,
            affine_matrix,
            offset=offset,
            output_shape=voxel.shape,
            mode='constant',
            cval=0.0,
            order=1,
        )
    else:
        # Fallback: manual interpolation with numpy
        result = np.zeros((h, w), dtype=voxel.dtype)
        # Clamp coordinates
        valid = (new_x >= 0) & (new_x < w) & (new_y >= 0) & (new_y < h)
        if valid.any():
            src_x = np.clip(new_x[valid].astype(np.float64), 0, w - 2)
            src_y = np.clip(new_y[valid].astype(np.float64), 0, h - 2)
            # Bilinear interpolation
            x0 = src_x.astype(int)
            y0 = src_y.astype(int)
            x1 = x0 + 1
            y1 = y0 + 1
            wa = (x1 - src_x) * (y1 - src_y)
            wb = (x1 - src_x) * (src_y - y0)
            wc = (src_x - x0) * (y1 - src_y)
            wd = (src_x - x0) * (src_y - y0)
            val = (wa * voxel[y0, x0] + wb * voxel[y0, x1] +
                   wc * voxel[y1, x0] + wd * voxel[y1, x1])
            result[valid] = val

    return result


def blend_images(voxel1, voxel2_transformed, alpha=0.5):
    '''Blend two images with given alpha.'''
    return (1 - alpha) * voxel1 + alpha * voxel2_transformed


# --- Pre-compute blended images for all solutions ---
print('Computing blended images...')
blended_images = {i: [] for i in range(len(angle_list)) if i in rotation_to_translations}
blended_info = {i: [] for i in range(len(angle_list)) if i in rotation_to_translations}

for idx in blended_images:
    solutions = rotation_to_translations[idx]
    angle = angle_list[idx]
    print(f'  Processing rotation peak {idx} (angle={angle:.4f} rad, {len(solutions)} solutions)...')

    for sol in solutions:
        matrix = sol.get('matrix')
        if matrix is None:
            # Reconstruct from angle
            matrix = np.eye(4)
            R2d = compute_2d_rotation_matrix(sol['angle'])
            matrix[:2, :2] = R2d
            matrix[:2, 3] = sol.get('translation', np.array([0, 0]))[:2]

        # Apply transformation
        try:
            transformed = apply_transformation_to_voxel(input_voxel2, matrix)
            blended = blend_images(input_voxel1, transformed, alpha=0.5)
            blended_images[idx].append(blended)
            blended_info[idx].append({
                'sol_num': sol['num'],
                'correlation': sol.get('correlation'),
            })
        except Exception as e:
            blended_images[idx].append(None)
            blended_info[idx].append({
                'sol_num': sol['num'],
                'error': str(e),
            })

print('Done computing blended images.')

# Count total images
total = sum(len(v) for v in blended_images.values())
print(f'Total blended images: {total}')


# --- Interactive blended image viewer ---
rot_slider = IntSlider(
    value=0,
    min=0,
    max=max(len(blended_images) - 1, 0),
    step=1,
    description='Rotation:',
    continuous_update=False,
)
trans_slider = IntSlider(
    value=0,
    min=0,
    max=len(blended_images.get(0, [])) - 1 if blended_images.get(0) else 0,
    step=1,
    description='Translation:',
    continuous_update=False,
)
info_label = Label('')

def update_trans_slider(Change):
    rot_idx = rot_slider.value
    n_trans = len(blended_images.get(rot_idx, []))
    trans_slider.max = max(n_trans - 1, 0)
    trans_slider.value = min(trans_slider.value, trans_slider.max)

rot_slider.observe(update_trans_slider, names='value')


def show_blended_image(rot_idx, trans_idx):
    if rot_idx not in blended_images or not blended_images[rot_idx]:
        fig = go.Figure()
        fig.add_annotation(text='No blended images available',
                         xref='paper', yref='paper', showarrow=False,
                         font=dict(size=20))
        fig.update_layout(height=500, width=500)
        fig.show()
        info_label.value = ''
        return

    img = blended_images[rot_idx][trans_idx]
    info = blended_info[rot_idx][trans_idx]

    if img is None:
        fig = go.Figure()
        fig.add_annotation(text=f'Error for solution {info.get("sol_num", "?")}: {info.get("error", "unknown")}',
                         xref='paper', yref='paper', showarrow=False,
                         font=dict(size=16))
        fig.update_layout(height=500, width=500)
        fig.show()
        info_label.value = ''
        return

    fig = go.Figure()
    fig.add_trace(go.Heatmap(z=img, colorscale='Viridis', showscale=True))

    rot_angle = angle_list[rot_idx] if rot_idx < len(angle_list) else 0
    corr = info.get('correlation')
    corr_str = f', corr={corr:.2f}' if corr is not None else ''
    title = (f'Rotation Peak {rot_idx} (angle={rot_angle:.4f} rad) '
             f'x Translation Solution {trans_idx} '
             f'(sol_num={info.get("sol_num", "?")}{corr_str})')

    fig.update_layout(height=500, width=500, title_text=title)
    fig.update_xaxes(showticklabels=False, showgrid=False)
    fig.update_yaxes(showticklabels=False, showgrid=False, scaleanchor='x', scaleratio=1)
    fig.show()

    info_label.value = (f'Rotation Peak {rot_idx}: angle={rot_angle:.4f} rad | '
                       f'Translation Solution {trans_idx} | '
                       f'Solution #{info.get("sol_num", "?")}{corr_str}')


# display(VBox([info_label, HBox([rot_slider, trans_slider])]))
# show_blended_image(rot_slider.value, trans_slider.value)

from ipywidgets import interact_manual
interact_manual(show_blended_image, rot_idx=rot_slider, trans_idx=trans_slider)


Computing blended images...
  Processing rotation peak 0 (angle=5.2033 rad, 37 solutions)...
  Processing rotation peak 1 (angle=5.5469 rad, 29 solutions)...
  Processing rotation peak 2 (angle=6.0377 rad, 18 solutions)...
  Processing rotation peak 3 (angle=0.0000 rad, 24 solutions)...
  Processing rotation peak 4 (angle=0.3927 rad, 24 solutions)...
  Processing rotation peak 5 (angle=0.5890 rad, 35 solutions)...
  Processing rotation peak 6 (angle=0.9817 rad, 39 solutions)...
  Processing rotation peak 7 (angle=1.4235 rad, 33 solutions)...
  Processing rotation peak 8 (angle=2.0126 rad, 39 solutions)...
  Processing rotation peak 9 (angle=2.1108 rad, 29 solutions)...
  Processing rotation peak 10 (angle=2.8471 rad, 27 solutions)...
  Processing rotation peak 11 (angle=3.1907 rad, 40 solutions)...
  Processing rotation peak 12 (angle=3.4361 rad, 27 solutions)...
  Processing rotation peak 13 (angle=3.6816 rad, 22 solutions)...
  Processing rotation peak 14 (angle=4.5160 rad, 41 soluti

interactive(children=(IntSlider(value=0, continuous_update=False, description='Rotation:', max=16), IntSlider(…

<function __main__.show_blended_image(rot_idx, trans_idx)>

## Translation Solutions per Rotation Angle

Bar chart showing how many translation solutions were detected for each candidate rotation angle.

In [10]:
if data_meta and angle_data:
    fig = go.Figure()
    
    angles = [a['angle'] for a in angle_data]
    num_trans = [a['numTranslations'] for a in angle_data]
    angle_indices = [a['angleIndex'] for a in angle_data]
    
    fig.add_trace(go.Bar(
        x=angles,
        y=num_trans,
        text=[f'idx={idx}, n={n}' for idx, n in zip(angle_indices, num_trans)],
        textposition='outside',
        marker_color='steelblue',
        marker_line_color='darkblue',
        marker_line_width=1.5,
        name='Translation Solutions'
    ))
    
    fig.update_xaxes(title_text='Rotation Angle (rad)')
    fig.update_yaxes(title_text='Number of Translation Peaks')
    fig.update_layout(height=450, width=900, title_text='Translation Solutions per Rotation Angle')
    fig.show()
else:
    print('No dataForReadIn.csv data available')


## Transform Summary

Overview of all detected transforms across all rotation angles.

In [11]:
# Load dataForReadIn.csv for summary
if data_meta and angle_data:
    print(f'Number of rotation angles: {data_meta["numAngles"]}')
    print(f'Total solutions: {data_meta["numTotalSolutions"]}')
    print()
    print('Rotation Angle Summary:')
    print(f'{"Idx":>4} {"Angle (rad)":>12} {"# Trans":>8}')
    print('-' * 28)
    for a in angle_data:
        print(f'{a["angleIndex"]:>4} {a["angle"]:>12.4f} {a["numTranslations"]:>8}')
    print()
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(12, 4))
    indices = [a['angleIndex'] for a in angle_data]
    num_trans = [a['numTranslations'] for a in angle_data]
    ax.bar(indices, num_trans, color='steelblue', edgecolor='black')
    ax.set_title('Number of Translation Solutions per Rotation Angle')
    ax.set_xlabel('Rotation Angle Index')
    ax.set_ylabel('Number of Translation Peaks')
    ax.set_xticks(indices)
    fig.tight_layout()
    
    plt.show()
elif not data_meta:
    print('dataForReadIn.csv not found or could not be parsed')
else:
    print('dataForReadIn.csv loaded but angle data is empty')


Number of rotation angles: 17
Total solutions: 577

Rotation Angle Summary:
 Idx  Angle (rad)  # Trans
----------------------------
   0       5.2033       37
   1       5.5469       29
   2       6.0377       18
   3       0.0000       24
   4       0.3927       24
   5       0.5890       35
   6       0.9817       39
   7       1.4235       33
   8       2.0126       39
   9       2.1108       29
  10       2.8471       27
  11       3.1907       40
  12       3.4361       27
  13       3.6816       22
  14       4.5160       41
  15       4.7124       86
  16       4.9578       27



/tmp/ipykernel_53787/340420936.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
